In [ ]:
file_path = "/content/creditcard.csv.zip"
df = pd.read_csv(file_path, compression='zip')

NameError: name 'pd' is not defined

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)
from sklearn.utils import class_weight, shuffle

import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras import layers, models

# ============================================================
# Load Dataset
# ============================================================

file_path = "/content/creditcard.csv.zip"

df = pd.read_csv(file_path, compression="zip")

print("\nDataset Loaded Successfully")
print("Shape :", df.shape)

print("\nTarget Distribution")
print(df["Class"].value_counts())
print(df["Class"].value_counts(normalize=True))

print("\nFirst Five Rows")
print(df.head())

# ============================================================
# Data Preprocessing
# ============================================================

df = df.dropna()

X = df.drop(columns=["Class"]).values
y = df["Class"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

print("\nPreprocessed Data")
print("Feature Shape :", X.shape)
print("Label Distribution :", np.unique(y, return_counts=True))

# ============================================================
# Train Test Split
# ============================================================

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Shuffle training data

X_train_full, y_train_full = shuffle(
    X_train_full,
    y_train_full,
    random_state=42
)

# ============================================================
# Simulate Federated Clients
# ============================================================

def simulate_clients(x_data, y_data, num_clients=5):

    client_data = {}

    samples_per_client = len(x_data) // num_clients

    for i in range(num_clients):

        start = i * samples_per_client

        if i == num_clients - 1:
            end = len(x_data)
        else:
            end = start + samples_per_client

        client_data[i] = (
            x_data[start:end],
            y_data[start:end]
        )

    return client_data


client_data = simulate_clients(
    X_train_full,
    y_train_full,
    num_clients=5
)

# ============================================================
# Class Weights
# ============================================================

combined_labels = np.concatenate(
    [client_data[i][1] for i in client_data]
)

weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(combined_labels),
    y=combined_labels
)

class_weights = dict(enumerate(weights))

print("\nClass Weights")
print(class_weights)

# ============================================================
# ANN Model
# ============================================================

def create_ann_model(input_dim):

    model = models.Sequential([

        tf.keras.Input(shape=(input_dim,)),

        layers.Dense(64, activation="relu"),

        layers.Dense(32, activation="relu"),

        layers.Dense(1, activation="sigmoid")

    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ============================================================
# Federated ANN Training
# ============================================================

def federated_train_ann(global_model, client_data, rounds=5):

    for rnd in range(rounds):

        print(f"\nFederated ANN Round {rnd+1}")

        local_weights = []

        for client_id, (cx, cy) in client_data.items():

            local_model = create_ann_model(cx.shape[1])

            local_model.set_weights(global_model.get_weights())

            local_model.fit(
                cx,
                cy,
                epochs=1,
                batch_size=32,
                verbose=0,
                class_weight=class_weights
            )

            local_weights.append(
                local_model.get_weights()
            )

        averaged_weights = [
            np.mean(layer, axis=0)
            for layer in zip(*local_weights)
        ]

        global_model.set_weights(
            averaged_weights
        )

    return global_model

# ============================================================
# Federated LightGBM
# ============================================================

def federated_train_lgb(client_data):

    models_list = []

    for client_id, (cx, cy) in client_data.items():

        negative = np.sum(cy == 0)
        positive = np.sum(cy == 1)

        scale_pos_weight = (
            negative / positive
            if positive > 0
            else 1
        )

        params = {

            "objective": "binary",

            "metric": "auc",

            "learning_rate": 0.05,

            "num_leaves": 31,

            "feature_fraction": 0.9,

            "bagging_fraction": 0.8,

            "bagging_freq": 5,

            "verbosity": -1,

            "scale_pos_weight": scale_pos_weight

        }

        train_dataset = lgb.Dataset(
            cx,
            label=cy
        )

        model = lgb.train(
            params,
            train_dataset,
            num_boost_round=50
        )

        models_list.append(model)

    def predict(X_new):

        predictions = [
            model.predict(X_new)
            for model in models_list
        ]

        return np.mean(predictions, axis=0)

    return predict

# ============================================================
# Train ANN
# ============================================================

global_ann = create_ann_model(X.shape[1])

global_ann = federated_train_ann(
    global_ann,
    client_data
)

ann_probs = global_ann.predict(X_test).flatten()

ann_preds = (ann_probs > 0.30).astype(int)

print("\n================ ANN RESULTS ================\n")

print(classification_report(
    y_test,
    ann_preds,
    zero_division=0
))

print("ROC AUC :", roc_auc_score(
    y_test,
    ann_probs
))

print("Confusion Matrix")

print(confusion_matrix(
    y_test,
    ann_preds
))

fraud_index = np.where(ann_preds == 1)[0]

print("\nANN Predicted Fraud Cases :", len(fraud_index))

for idx in fraud_index[:10]:

    print(
        f"Index={idx}  "
        f"Probability={ann_probs[idx]:.4f}  "
        f"Actual={y_test[idx]}"
    )

# ============================================================
# Train LightGBM
# ============================================================

predict_fn = federated_train_lgb(
    client_data
)

lgb_probs = predict_fn(X_test)

lgb_preds = (lgb_probs > 0.50).astype(int)

print("\n================ LightGBM RESULTS ================\n")

print(classification_report(
    y_test,
    lgb_preds,
    zero_division=0
))

print("ROC AUC :", roc_auc_score(
    y_test,
    lgb_probs
))

print("Confusion Matrix")

print(confusion_matrix(
    y_test,
    lgb_preds
))

fraud_index = np.where(lgb_preds == 1)[0]

print("\nLightGBM Predicted Fraud Cases :", len(fraud_index))

for idx in fraud_index[:10]:

    print(
        f"Index={idx}  "
        f"Probability={lgb_probs[idx]:.4f}  "
        f"Actual={y_test[idx]}"
    )


Dataset Loaded Successfully
Shape : (284807, 31)

Target Distribution
Class
0    284315
1       492
Name: count, dtype: int64
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

First Five Rows
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.68